# Day 1 | Lab 1.3: Structured Outputs & Multi-Provider Comparison

**Duration:** ~1.5 hours

**Scenario.** Retail-banking AI customer-support evaluation ("Credit Card Dispute Intake", "Customer Intent Classification") — preserved from the B4 source notebook. Banking framing kept because the source is already banking; trainees can map this directly to the kinds of intake/triage flows they'll see at financial-services clients.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Configure API keys for 4+ LLM providers in a local-venv environment.
2. Call each provider's API with the same prompt and compare latency, token usage, and response quality side-by-side.
3. Distinguish **temperature** (creativity control for non-reasoning models) from **reasoning effort** (depth control for reasoning models).
4. Use **Pydantic models** with OpenAI's Responses API (`responses.parse`) to extract structured, schema-validated data from free-text customer messages.
5. Implement a reusable multi-provider fallback pattern for enterprise LLM evaluation.

**Lab Flow**

| Step | What We Do | Key Technology |
|---|---|---|
| 1 | Install SDKs & configure keys | OpenAI, Anthropic, Google GenAI, Groq |
| 2 | Send identical banking queries to 4 providers | Multi-provider comparison |
| 3 | Explore temperature vs. reasoning effort | `gpt-4.1-mini` vs `gpt-5-mini` |
| 4 | Extract structured data with Pydantic | OpenAI Responses API + `text_format` |
| 5 | Compare structured output across providers | Native vs prompt-based JSON |
| 6 | Build a multi-provider fallback | Production resilience pattern |

---


## 1. Install Dependencies

We install the official SDKs for all four providers. Each SDK handles authentication, request formatting, and response parsing for its respective API.

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install openai anthropic google-genai groq pydantic -q


## 2. API Key Configuration

We use Google Colab's **Secrets Manager** to load API keys securely — never hardcode keys in notebooks.

> **Setup:** Go to the 🔑 icon in Colab's left sidebar → Add each key (`OPENAI_API_KEY`, `GROQ_API_KEY`, `GEMINI_API_KEY`, `ANTHROPIC_API_KEY`) → Toggle "Notebook access" ON for each.

In [1]:
import os
env_path = r"C:\Users\prash\OneDrive\2026\Training\eClerx-GenAI-Training\.env"

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv(env_path)
except ImportError:
    pass

# Verify keys are loaded (prints status, never prints actual values)
for key in ['OPENAI_API_KEY', 'GROQ_API_KEY', 'GEMINI_API_KEY', 'ANTHROPIC_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


OPENAI_API_KEY: ✅ Loaded
GROQ_API_KEY: ✅ Loaded
GEMINI_API_KEY: ✅ Loaded
ANTHROPIC_API_KEY: ✅ Loaded


## 3. Initialize Clients & Helper Functions

Each provider has a different SDK pattern. We initialize all clients upfront and create a unified helper function for side-by-side comparisons.

In [2]:
import time
import json
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from groq import Groq

# Initialize all provider clients
openai_client = OpenAI()
anthropic_client = Anthropic()
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
groq_client = Groq()

print("All 4 provider clients initialized ✅")

All 4 provider clients initialized ✅


In [3]:
def call_openai(prompt, system_prompt="You are a helpful banking assistant.", model="gpt-4.1-mini", temperature=0.7):
    """Call OpenAI Chat Completions API (non-reasoning model with temperature)."""
    start = time.time()
    response = openai_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    latency = round(time.time() - start, 2)
    text = response.choices[0].message.content
    tokens = response.usage.total_tokens if response.usage else 0
    return {"text": text, "latency": latency, "tokens": tokens, "model": model}


def call_openai_reasoning(prompt, model="gpt-5-mini", reasoning_effort="medium"):
    """Call OpenAI Responses API (reasoning model — no temperature)."""
    start = time.time()
    response = openai_client.responses.create(
        model=model,
        reasoning={"effort": reasoning_effort},
        input=prompt
    )
    latency = round(time.time() - start, 2)
    text = response.output_text
    return {"text": text, "latency": latency, "tokens": "N/A (Responses API)", "model": model}


def call_anthropic(prompt, system_prompt="You are a helpful banking assistant.", model="claude-haiku-4-5", temperature=0.7):
    """Call Anthropic Claude API."""
    start = time.time()
    response = anthropic_client.messages.create(
        model=model,
        max_tokens=1024,
        temperature=temperature,
        system=system_prompt,
        messages=[{"role": "user", "content": prompt}]
    )
    latency = round(time.time() - start, 2)
    text = response.content[0].text
    tokens = response.usage.input_tokens + response.usage.output_tokens
    return {"text": text, "latency": latency, "tokens": tokens, "model": model}


def call_gemini(prompt, system_prompt="You are a helpful banking assistant.", model="gemini-2.5-flash", temperature=0.7):
    """Call Google Gemini API."""
    start = time.time()
    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config=genai.types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=1024,
            system_instruction=system_prompt
        )
    )
    latency = round(time.time() - start, 2)
    text = response.text
    tokens = response.usage_metadata.total_token_count if response.usage_metadata else 0
    return {"text": text, "latency": latency, "tokens": tokens, "model": model}


def call_groq(prompt, system_prompt="You are a helpful banking assistant.", model="llama-3.3-70b-versatile", temperature=0.7):
    """Call Groq Cloud API (OpenAI-compatible, runs open-source Llama)."""
    start = time.time()
    response = groq_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    latency = round(time.time() - start, 2)
    text = response.choices[0].message.content
    tokens = response.usage.total_tokens if response.usage else 0
    return {"text": text, "latency": latency, "tokens": tokens, "model": model}


def display_result(result, provider_name):
    """Pretty-print a provider result."""
    print(f"\n{'=' * 60}")
    print(f"📋 {provider_name} ({result['model']})")
    print(f"⏱️ Latency: {result['latency']}s | 🔢 Tokens: {result['tokens']}")
    print(f"{'-' * 60}")
    print(result['text'])  # [:500]
    if len(result['text']) > 500:
        print(f"\n... [{len(result['text'])} chars total]")

print("Helper functions defined ✅")

Helper functions defined ✅


In [4]:
model="gpt-5-mini"
reasoning_effort="medium"
customer_query = """I want to dispute a charge of ₹18,250 on my credit card from last Tuesday.
I don't recognize the merchant 'QuickMart Online' and I never authorized this transaction.
My card was in my possession the whole time. Please help me resolve this immediately."""

system_prompt = """You are a senior customer support agent at HDFC Bank.
Respond to the customer's concern professionally. Acknowledge their issue,
outline the next steps for resolution, and provide estimated timelines.
Keep your response concise (under 150 words)."""

response = openai_client.responses.create(
    model=model,
    reasoning={"effort": reasoning_effort},
    input=system_prompt + customer_query
)

In [5]:
response

Response(id='resp_0dc32e2e609eaf250069f1653dca4c8195aa2e6b01178f3a34', created_at=1777427773.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-mini-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_0dc32e2e609eaf250069f1653ef234819594fe72fdf3565068', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0dc32e2e609eaf250069f1654c08f881958a14587d596dbc5f', content=[ResponseOutputText(annotations=[], text='Sorry to hear about the unauthorised ₹18,250 charge. I will help you escalate this immediately.\n\nNext steps:\n1) Confirm if you want me to block your card now and arrange a replacement.\n2) I will raise a dispute/chargeback with our investigations team—please reply with the last 4 digits of the card, exact transaction date/time (you said last Tuesday), and any SMS/statement screenshot or transaction reference.\n3) We will advise any signed dispute form or ID documents neede

In [6]:
print(response.output_text)  # this is the ACTUAL LLM's reply

Sorry to hear about the unauthorised ₹18,250 charge. I will help you escalate this immediately.

Next steps:
1) Confirm if you want me to block your card now and arrange a replacement.
2) I will raise a dispute/chargeback with our investigations team—please reply with the last 4 digits of the card, exact transaction date/time (you said last Tuesday), and any SMS/statement screenshot or transaction reference.
3) We will advise any signed dispute form or ID documents needed.

Estimated timelines:
- Provisional credit (if eligible) usually within 7–10 working days after submission.
- Full investigation and final resolution typically complete within 45–90 days.

Reply with the requested details or call our 24x7 support to proceed immediately.


## 4. Multi-Provider Comparison — Same Banking Query

### Business Scenario: Credit Card Dispute Intake

HDFC Bank receives thousands of customer messages daily via their chatbot. Before routing a message to the right team, the AI assistant must understand the intent and provide an initial response. We'll send the **same customer message** to all 4 providers and compare their responses.

> *"I want to dispute a charge of ₹18,250 on my credit card from last Tuesday. I don't recognize the merchant 'QuickMart Online' and I never authorized this transaction. My card was in my possession the whole time. Please help me resolve this immediately."*

In [ ]:
# Same customer query to all 4 providers
customer_query = """I want to dispute a charge of ₹18,250 on my credit card from last Tuesday.
I don't recognize the merchant 'QuickMart Online' and I never authorized this transaction.
My card was in my possession the whole time. Please help me resolve this immediately."""

system_prompt = """You are a senior customer support agent at HDFC Bank.
Respond to the customer's concern professionally. Acknowledge their issue,
outline the next steps for resolution, and provide estimated timelines.
Keep your response concise (under 150 words)."""

# --- Call all 4 providers ---
results = {}
results["OpenAI"] = call_openai(customer_query, system_prompt)
results["Anthropic"] = call_anthropic(customer_query, system_prompt)
results["Gemini"] = call_gemini(customer_query, system_prompt)
results["Groq (Llama)"] = call_groq(customer_query, system_prompt)

# Display all results
for provider, result in results.items():
    display_result(result, provider)


📋 OpenAI (gpt-4.1-mini)
⏱️ Latency: 2.41s | 🔢 Tokens: 224
------------------------------------------------------------
Dear Customer,

Thank you for reaching out. We understand your concern regarding the unauthorized ₹18,250 charge from 'QuickMart Online.' To resolve this, we will initiate a dispute investigation and temporarily block further transactions on your card for security.

Please confirm your card number's last four digits and provide any additional details if available. Our team will review the transaction and update you within 7 business days. Meanwhile, we recommend monitoring your account for any other unusual activity.

Rest assured, we are committed to resolving this promptly.

Best regards,  
HDFC Bank Customer Support

... [626 chars total]

📋 Anthropic (claude-haiku-4-5)
⏱️ Latency: 2.64s | 🔢 Tokens: 331
------------------------------------------------------------
Thank you for bringing this to our attention. I understand your concern about the unauthorized charge o

In [ ]:
# Side-by-side metrics comparison
print("\n📊 PROVIDER COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Provider':<18} {'Model':<28} {'Latency':>8} {'Tokens':>8}")
print("-" * 70)
for provider, result in results.items():
    print(f"{provider:<18} {result['model']:<28} {result['latency']:>7}s {str(result['tokens']):>8}")
print("=" * 70)
print("\n💡 Groq typically shows lowest latency (custom inference hardware).")
print("💡 Token counts vary because each provider uses different tokenizers.")


📊 PROVIDER COMPARISON SUMMARY
Provider           Model                         Latency   Tokens
----------------------------------------------------------------------
OpenAI             gpt-4.1-mini                    2.41s      224
Anthropic          claude-haiku-4-5                2.64s      331
Gemini             gemini-2.5-flash                3.43s      700
Groq (Llama)       llama-3.3-70b-versatile         1.27s      251

💡 Groq typically shows lowest latency (custom inference hardware).
💡 Token counts vary because each provider uses different tokenizers.


## 5. Temperature vs. Reasoning Effort

This is one of the most important concepts to understand when working with modern LLMs.

| Concept | Applies To | Controls | Range |
|---------|-----------|----------|-------|
| **Temperature** | Non-reasoning models (`gpt-4.1-mini`, `claude-haiku-4-5`, `llama-3.3-70b`) | Randomness / creativity of output | 0.0 (deterministic) → 2.0 (very random) |
| **Reasoning Effort** | Reasoning models (`gpt-5-mini`) | Depth of internal chain-of-thought | `low` / `medium` / `high` |

**Key rule:** Reasoning models like `gpt-5-mini` do **NOT** support the `temperature` parameter. They decide their own "creativity level" internally based on the task. If you need to control creativity, use a non-reasoning model like `gpt-4.1-mini`.

### Business Scenario: Investment Summary Generation

A wealth management team wants to understand how output variability affects client-facing documents. Deterministic outputs are preferred for compliance reports (low temperature), while marketing drafts benefit from creativity (high temperature).

In [ ]:
# Temperature effects with gpt-4.1-mini (non-reasoning model)
prompt = """Write a 2-sentence summary of why diversification matters
for a retail banking customer's investment portfolio."""

temperatures = [0.1, 0.5, 0.9]

print("🌡️ TEMPERATURE COMPARISON — gpt-4.1-mini")
print("=" * 60)

for temp in temperatures:
    result = call_openai(prompt, temperature=temp)
    print(f"\n🔹 Temperature = {temp}")
    print(f"   {result['text']}")
    print(f"   ⏱️ {result['latency']}s")

🌡️ TEMPERATURE COMPARISON — gpt-4.1-mini

🔹 Temperature = 0.1
   Diversification matters for a retail banking customer's investment portfolio because it helps spread risk across different asset types, reducing the impact of any single investment's poor performance. This strategy enhances the potential for more stable and consistent returns over time.
   ⏱️ 1.72s

🔹 Temperature = 0.5
   Diversification matters for a retail banking customer's investment portfolio because it helps spread risk across different asset types, reducing the impact of any single investment's poor performance. This strategy enhances the potential for more stable and consistent returns over time.
   ⏱️ 0.85s

🔹 Temperature = 0.9
   Diversification matters for a retail banking customer's investment portfolio because it helps spread risk across different asset types, reducing the impact of any single investment's poor performance. This approach enhances the potential for more stable and consistent returns over time.

In [ ]:
# Run the SAME prompt 3 times at temperature=0.9 to show variability
print("🎲 VARIABILITY TEST — Same prompt, 3 runs at temperature=0.9")
print("=" * 60)

for i in range(3):
    result = call_openai(prompt, temperature=0.9)
    print(f"\n🔹 Run {i+1}: {result['text'][:200]}")

print("\n💡 Notice how each response is different — that's high temperature at work.")

🎲 VARIABILITY TEST — Same prompt, 3 runs at temperature=0.9

🔹 Run 1: Diversification matters for a retail banking customer's investment portfolio because it helps spread risk across different asset types, reducing the impact of any single investment's poor performance.

🔹 Run 2: Diversification matters for a retail banking customer's investment portfolio because it helps spread risk across different asset types, reducing the impact of any single investment's poor performance.

🔹 Run 3: Diversification matters for a retail banking customer's investment portfolio because it helps spread risk across different asset types, reducing the impact of any single investment's poor performance.

💡 Notice how each response is different — that's high temperature at work.


In [ ]:
# Now compare with gpt-5-mini (reasoning model — NO temperature)
print("🧠 REASONING MODEL — gpt-5-mini (Responses API)")
print("=" * 60)

reasoning_prompt = f"""You are a senior banking analyst. {prompt}
Also explain the mathematical concept behind risk reduction through diversification."""

for effort in ["low", "medium", "high"]:
    result = call_openai_reasoning(reasoning_prompt, reasoning_effort=effort)
    print(f"\n🔹 Reasoning Effort = '{effort}'")
    print(f"   {result['text'][:300]}")
    print(f"   ⏱️ {result['latency']}s")

print("\n💡 Higher reasoning effort = deeper analysis but slower response.")
print("💡 Unlike temperature, reasoning effort doesn't add randomness — it adds depth.")

🧠 REASONING MODEL — gpt-5-mini (Responses API)

🔹 Reasoning Effort = 'low'
   Diversification matters because spreading investments across different assets reduces the impact of any single loss, improving the stability of returns for a retail banking customer's portfolio while preserving expected return. It also lowers overall portfolio risk more effectively than concentratin
   ⏱️ 9.41s

🔹 Reasoning Effort = 'medium'
   Diversification matters because spreading money across many different assets reduces the impact of any one investment’s poor performance on the overall portfolio, improving return stability without necessarily lowering expected returns; holding assets that don’t move perfectly together smooths retur
   ⏱️ 9.97s

🔹 Reasoning Effort = 'high'
   Diversification matters because spreading investments across different asset classes, sectors and geographies reduces the chance that any one holding’s poor performance will materially damage a retail investor’s portfolio. By comb

### 🔍 Discussion Points

- **Temperature** adds randomness — useful for creative tasks. Low temperature (0.1) gives near-identical outputs across runs. High temperature (0.9) produces varied outputs each time.
- **Reasoning effort** adds depth — the model "thinks harder" at `high` effort, producing more thorough analysis. But the output is still deterministic for the same input.
- **When to use which?** Compliance reports → `gpt-4.1-mini` at `temperature=0.1`. Creative marketing → `gpt-4.1-mini` at `temperature=0.8`. Complex financial analysis → `gpt-5-mini` with `reasoning_effort="high"`.

---

## 6. Structured Output with Pydantic

### Business Scenario: Customer Intent Classification at Scale

HDFC Bank processes 50,000+ customer messages daily. Instead of free-text responses, the intake system needs **structured, machine-readable classifications** that can be directly fed into routing rules, dashboards, and CRM systems. We use Pydantic models to define the exact schema we expect from the LLM, and OpenAI's Responses API guarantees the output conforms to it.

This is fundamentally different from asking the LLM to "return JSON" — structured outputs use **constrained decoding** to ensure 100% schema compliance at the token generation level.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Define the structured output schema using Pydantic
class CustomerIntent(BaseModel):
    """Schema for classifying customer banking messages."""
    category: Literal["dispute", "inquiry", "complaint", "request", "fraud_report"] = Field(
        description="Primary intent category of the customer message"
    )
    urgency: Literal["low", "medium", "high", "critical"] = Field(
        description="How urgently this needs attention"
    )
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer's emotional tone"
    )
    entities: list[str] = Field(
        description="Key entities extracted: amounts, dates, merchant names, card types"
    )
    summary: str = Field(
        description="One-sentence summary of the customer's core issue"
    )
    recommended_action: str = Field(
        description="Suggested next step for the support agent"
    )

print("CustomerIntent schema defined ✅")
print(f"Fields: {list(CustomerIntent.model_fields.keys())}")

CustomerIntent schema defined ✅
Fields: ['category', 'urgency', 'sentiment', 'entities', 'summary', 'recommended_action']


In [ ]:
# Use OpenAI Responses API with structured output (text_format)
test_message = """I want to dispute a charge of ₹18,250 on my credit card from last Tuesday.
I don't recognize the merchant 'QuickMart Online' and I never authorized this transaction.
My card was in my possession the whole time. Please help me resolve this immediately."""

start = time.time()
response = openai_client.responses.parse(
    model="gpt-5-mini",
    input=[
        {"role": "system", "content": "You are an intent classifier for HDFC Bank customer messages. Classify the message accurately."},
        {"role": "user", "content": test_message}
    ],
    text_format=CustomerIntent, # attaching the Pydantic Schema for the  model to refer
)
latency = round(time.time() - start, 2)

# The response is a validated Pydantic object — guaranteed to match our schema
intent = response.output_parsed

print(f"📋 STRUCTURED CLASSIFICATION (⏱️ {latency}s)")
print("=" * 60)
print(f"  Category:     {intent.category}")
print(f"  Urgency:      {intent.urgency}")
print(f"  Sentiment:    {intent.sentiment}")
print(f"  Entities:     {intent.entities}")
print(f"  Summary:      {intent.summary}")
print(f"  Action:       {intent.recommended_action}")
print("\n✅ Output is a validated Pydantic object — no JSON parsing needed!")

📋 STRUCTURED CLASSIFICATION (⏱️ 13.96s)
  Category:     dispute
  Urgency:      critical
  Sentiment:    angry
  Entities:     ['₹18,250', 'last Tuesday', 'QuickMart Online', 'credit card']
  Summary:      Customer disputes an unauthorized ₹18,250 credit card charge from 'QuickMart Online' on last Tuesday and requests immediate resolution, stating the card was in their possession.
  Action:       Open an urgent dispute and fraud investigation: (1) Immediately block/disable the card for further transactions and offer replacement; (2) Initiate chargeback/provisional credit per bank policy and file a fraud case; (3) Collect evidence: transaction date/time, last 4 card digits, statement screenshot, any merchant communications; (4) Provide the customer with a case/reference number and expected timeline for resolution; (5) Escalate to fraud/collections team if required.

✅ Output is a validated Pydantic object — no JSON parsing needed!


In [ ]:
intent

CustomerIntent(category='dispute', urgency='critical', sentiment='angry', entities=['₹18,250', 'last Tuesday', 'QuickMart Online', 'credit card'], summary="Customer disputes an unauthorized ₹18,250 credit card charge from 'QuickMart Online' on last Tuesday and requests immediate resolution, stating the card was in their possession.", recommended_action='Open an urgent dispute and fraud investigation: (1) Immediately block/disable the card for further transactions and offer replacement; (2) Initiate chargeback/provisional credit per bank policy and file a fraud case; (3) Collect evidence: transaction date/time, last 4 card digits, statement screenshot, any merchant communications; (4) Provide the customer with a case/reference number and expected timeline for resolution; (5) Escalate to fraud/collections team if required.')

In [ ]:
# Classify 5 diverse banking customer messages
test_messages = [
    "What is the current interest rate on your 2-year fixed deposit? I have ₹10,00,000 to invest.",
    "Your ATM at MG Road, Bangalore has been out of service for 3 days now! This is unacceptable. I've complained twice already.",
    "I'd like to increase my credit card limit from ₹2,00,000 to ₹5,00,000. My salary was recently revised to ₹25,00,000 per annum.",
    "URGENT: I just received an OTP I didn't request. Someone may be trying to access my account. My account number is XXXX1234.",
    "Can you help me set up a standing instruction for my monthly SIP of ₹15,000 to Axis Mutual Fund?"
]

print("📋 BATCH CLASSIFICATION — 5 Banking Messages")
print("=" * 70)

for i, msg in enumerate(test_messages, 1):
    response = openai_client.responses.parse(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": "You are an intent classifier for HDFC Bank customer messages. Classify accurately."},
            {"role": "user", "content": msg}
        ],
        text_format=CustomerIntent,
    )
    intent = response.output_parsed

    print(f"\n{'─' * 70}")
    print(f"Message {i}: {msg[:80]}...")
    print(f"  → Category: {intent.category:<15} Urgency: {intent.urgency:<10} Sentiment: {intent.sentiment}")
    print(f"  → Entities: {intent.entities}")
    print(f"  → Action: {intent.recommended_action}")

print(f"\n{'=' * 70}")
print("✅ All 5 messages classified with guaranteed schema compliance")

📋 BATCH CLASSIFICATION — 5 Banking Messages

──────────────────────────────────────────────────────────────────────
Message 1: What is the current interest rate on your 2-year fixed deposit? I have ₹10,00,00...
  → Category: inquiry         Urgency: low        Sentiment: neutral
  → Entities: ['₹10,00,000', '2-year fixed deposit', 'interest rate']
  → Action: Provide the bank's current 2-year fixed deposit interest rate (separately for regular and senior citizens if applicable), note any differences for NRE/NRO accounts, mention tax/TDS and premature withdrawal rules, and ask if they want to proceed with opening the FD or need help with online application.

──────────────────────────────────────────────────────────────────────
Message 2: Your ATM at MG Road, Bangalore has been out of service for 3 days now! This is u...
  → Category: complaint       Urgency: high       Sentiment: angry
  → Entities: ['ATM', 'MG Road, Bangalore', '3 days', 'complained twice']
  → Action: Apologize, log/

## 7. Multi-Provider Structured Output Comparison

Not all providers support constrained structured output natively. Let's compare approaches:

| Provider | Native Structured Output | Method |
|----------|:---:|--------|
| OpenAI (`gpt-5-mini`) | ✅ Yes | `responses.parse()` with `text_format=PydanticModel` |
| OpenAI (`gpt-4.1-mini`) | ✅ Yes | `chat.completions.create()` with `response_format` |
| Anthropic (Claude) | ❌ No | Prompt-based JSON extraction + manual Pydantic validation |
| Google Gemini | ✅ Yes | `generate_content()` with `response_schema` |
| Groq (Llama) | ⚠️ Partial | `response_format={"type": "json_object"}` + prompt engineering |

Let's test the prompt-based approach with Anthropic and Groq, and compare reliability.

In [ ]:
# Define the JSON schema as a prompt instruction (for non-native providers)
json_schema_prompt = """Classify this banking customer message. Return ONLY valid JSON matching this exact schema:
{
  "category": "dispute|inquiry|complaint|request|fraud_report",
  "urgency": "low|medium|high|critical",
  "sentiment": "positive|neutral|negative|angry",
  "entities": ["list", "of", "key", "entities"],
  "summary": "one sentence summary",
  "recommended_action": "suggested next step"
}

Customer message: I want to dispute a charge of ₹18,250 on my credit card from last Tuesday.
I don't recognize the merchant 'QuickMart Online' and I never authorized this transaction."""

# --- Anthropic (prompt-based JSON) ---
print("📋 ANTHROPIC — Prompt-Based JSON Extraction")
print("-" * 50)
result = call_anthropic(json_schema_prompt, system_prompt="You are a JSON classifier. Return ONLY valid JSON, no other text.")
try:
    parsed = json.loads(result["text"])
    validated = CustomerIntent(**parsed)
    print(f"  Category: {validated.category} | Urgency: {validated.urgency}")
    print(f"  ✅ Successfully parsed and validated!")
except (json.JSONDecodeError, Exception) as e:
    print(f"  ❌ Parse/validation error: {e}")
    print(f"  Raw: {result['text'][:200]}")

# --- Groq / Llama (JSON mode) ---
print("\n📋 GROQ (Llama) — JSON Mode")
print("-" * 50)
start = time.time()
response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    response_format={"type": "json_object"},
    messages=[
        {"role": "system", "content": "You are a JSON classifier. Return ONLY valid JSON."},
        {"role": "user", "content": json_schema_prompt}
    ]
)
groq_text = response.choices[0].message.content
latency = round(time.time() - start, 2)
try:
    parsed = json.loads(groq_text)
    validated = CustomerIntent(**parsed)
    print(f"  Category: {validated.category} | Urgency: {validated.urgency}")
    print(f"  ✅ Successfully parsed and validated! (⏱️ {latency}s)")
except (json.JSONDecodeError, Exception) as e:
    print(f"  ❌ Parse/validation error: {e}")
    print(f"  Raw: {groq_text[:200]}")

print("\n💡 Native structured output (OpenAI) guarantees schema compliance.")
print("💡 Prompt-based extraction works but may occasionally fail validation.")

📋 ANTHROPIC — Prompt-Based JSON Extraction
--------------------------------------------------
  ❌ Parse/validation error: Expecting value: line 1 column 1 (char 0)
  Raw: ```json
{
  "category": "dispute",
  "urgency": "high",
  "sentiment": "negative",
  "entities": ["₹18,250", "credit card", "QuickMart Online", "last Tuesday", "unauthorized transaction"],
  "summary"

📋 GROQ (Llama) — JSON Mode
--------------------------------------------------
  Category: dispute | Urgency: high
  ✅ Successfully parsed and validated! (⏱️ 0.59s)

💡 Native structured output (OpenAI) guarantees schema compliance.
💡 Prompt-based extraction works but may occasionally fail validation.


## 8. Cost & Latency Analysis

Understanding the cost-performance trade-off is critical for enterprise LLM deployment decisions.

In [ ]:
# Approximate pricing per 1M tokens (March 2026 estimates)
pricing = {
    "gpt-4.1-mini":            {"input": 0.40,  "output": 1.60,  "notes": "Best non-reasoning model"},
    "gpt-5-mini":              {"input": 1.10,  "output": 4.40,  "notes": "Reasoning model, highest quality"},
    "claude-haiku-4-5":        {"input": 0.80,  "output": 4.00,  "notes": "Anthropic's fast model"},
    "gemini-2.5-flash":        {"input": 0.15,  "output": 0.60,  "notes": "Google's cost leader"},
    "llama-3.3-70b (Groq)":    {"input": 0.59,  "output": 0.79,  "notes": "Open-source, fastest inference"},
}

print("💰 APPROXIMATE PRICING (per 1M tokens, USD)")
print("=" * 80)
print(f"{'Model':<28} {'Input':>8} {'Output':>8} {'Notes'}")
print("-" * 80)
for model, info in pricing.items():
    print(f"{model:<28} ${info['input']:>6.2f}  ${info['output']:>6.2f}  {info['notes']}")
print("=" * 80)
print("\n💡 For high-volume intake classification (50K msgs/day):")
print("   → Gemini 2.5 Flash offers best cost efficiency")
print("   → Groq/Llama offers fastest inference + open-source flexibility")
print("   → gpt-5-mini offers best accuracy for complex cases")
print("   → Strategy: Use cheap/fast model for routing, reasoning model for complex cases")

💰 APPROXIMATE PRICING (per 1M tokens, USD)
Model                           Input   Output Notes
--------------------------------------------------------------------------------
gpt-4.1-mini                 $  0.40  $  1.60  Best non-reasoning model
gpt-5-mini                   $  1.10  $  4.40  Reasoning model, highest quality
claude-haiku-4-5             $  0.80  $  4.00  Anthropic's fast model
gemini-2.5-flash             $  0.15  $  0.60  Google's cost leader
llama-3.3-70b (Groq)         $  0.59  $  0.79  Open-source, fastest inference

💡 For high-volume intake classification (50K msgs/day):
   → Gemini 2.5 Flash offers best cost efficiency
   → Groq/Llama offers fastest inference + open-source flexibility
   → gpt-5-mini offers best accuracy for complex cases
   → Strategy: Use cheap/fast model for routing, reasoning model for complex cases


## 9. Production Pattern: Multi-Provider Fallback

In production, you never rely on a single provider. Here's a reusable pattern that tries providers in order and falls back on failure.

In [ ]:
def classify_with_fallback(message, providers=None):
    """Try multiple providers in order; return first successful classification.

    Args:
        message: Customer message to classify
        providers: List of (name, callable) tuples. Defaults to OpenAI → Groq → Anthropic.

    Returns:
        dict with provider name, classification result, and latency
    """
    if providers is None:
        providers = [
            ("OpenAI (gpt-5-mini)", lambda msg: openai_client.responses.parse(
                model="gpt-5-mini",
                input=[{"role": "system", "content": "Classify this banking message."}, {"role": "user", "content": msg}],
                text_format=CustomerIntent
            ).output_parsed),
            ("Groq (Llama)", lambda msg: CustomerIntent(**json.loads(
                groq_client.chat.completions.create(
                    model="llama-3.3-70b-versatile", temperature=0.1,
                    response_format={"type": "json_object"},
                    messages=[{"role": "system", "content": "Return JSON with keys: category, urgency, sentiment, entities, summary, recommended_action"},
                              {"role": "user", "content": msg}]
                ).choices[0].message.content
            ))),
        ]

    for name, classify_fn in providers:
        try:
            start = time.time()
            result = classify_fn(message)
            latency = round(time.time() - start, 2)
            return {"provider": name, "result": result, "latency": latency, "status": "✅ success"}
        except Exception as e:
            print(f"  ⚠️ {name} failed: {e}")
            continue

    return {"provider": "none", "result": None, "latency": 0, "status": "❌ all providers failed"}

# Test the fallback pattern
test_msg = "I need to block my debit card immediately — I think it was stolen at the metro station."
output = classify_with_fallback(test_msg)

print(f"Provider: {output['provider']}")
print(f"Status: {output['status']} (⏱️ {output['latency']}s)")
if output['result']:
    print(f"Category: {output['result'].category} | Urgency: {output['result'].urgency}")
    print(f"Summary: {output['result'].summary}")

Provider: OpenAI (gpt-5-mini)
Status: ✅ success (⏱️ 4.79s)
Category: fraud_report | Urgency: critical
Summary: Customer believes their debit card was stolen at a metro station and asks to block it immediately.


---

## 10. Conclusion & Key Takeaways

### What We Covered

| Task | Key Takeaway |
|---|---|
| **Multi-provider setup** | Each provider has its own SDK pattern, but a thin wrapper hides those differences |
| **Side-by-side comparison** | Latency, token usage, and response quality vary significantly across providers |
| **Temperature vs. reasoning** | Temperature controls randomness (non-reasoning); reasoning effort controls depth (`gpt-5-mini`) |
| **Pydantic structured output** | OpenAI's `responses.parse(text_format=...)` guarantees schema compliance — no JSON-parse failures |
| **Multi-provider structured** | Native structured output (OpenAI/Gemini) is more reliable than prompt-based JSON extraction |
| **Fallback pattern** | Production systems should try multiple providers in sequence for resilience |

### Connection to Lab 1.1
Lab 1.1 introduced a **provider-agnostic wrapper** function `call_<provider>(system, user, **kwargs) → dict`. This lab extends that pattern in two ways:
1. **Schema enforcement.** Replace the free-text return value with a Pydantic model — `responses.parse(..., text_format=MyModel)` — so downstream code consumes typed data, not strings to be regex'd.
2. **Fallback chaining.** The `classify_with_fallback(...)` pattern is the same wrapper interface, just called in a `for provider in providers: try: ... except: continue` loop.

Pydantic + the unified-wrapper pattern together give you a **typed, swappable LLM call** — the foundation for every chain we'll build from Module 3 onward.

## 11. Stretch Exercise (Optional)
1. **5th provider.** Add Mistral or Cohere to the comparison and observe how the same Pydantic schema travels (or doesn't).
2. **`LoanApplication` model.** Build a Pydantic schema with `loan_type`, `amount`, `tenure_years`, `applicant_risk`, `notes` and classify 10 loan-inquiry messages.
3. **Latency benchmark.** Call each provider 10× and compute mean / P95 latency.
4. **Hybrid pipeline.** Use Gemini Flash (cheap) for first-pass intake classification, escalate to `gpt-5-mini` (high) only when the first pass returns `confidence < 0.7`.
5. **Strict-mode JSON.** Try OpenAI's `response_format={'type': 'json_schema', ...}` strict mode and compare reliability vs `responses.parse`.

---

**Next Lab:** Lab 1.4 — Transformer Architecture & Tokenization Fundamentals 🧠
